# 02 — Re-evaluate the released full-model checkpoints (no training)
Scope of this notebook: it verifies **only the three full dual-graph checkpoints** (seeds 42/123/2026) — their SHA-256, stored metadata, validation loss, locked-test/HD pooled MSE and two-hop masking. It does not verify ablations, PPI controls, specificity panels or structure preservation (see `docs/source_cell_mapping.md` for their status).
The two-hop stress test uses the existing checkpoints; nothing is retrained.

In [ ]:
# --- Setup: mount Drive, unpack the repository, install dependencies ---
from google.colab import drive
drive.mount('/content/drive')
import subprocess, sys, os
BASE = '/content/drive/MyDrive/New-SASP-idea'                 # data/checkpoints live here (not in the repo)
REPO_ZIP = BASE + '/code_release_export_2026-10-23/spatial-sasp-gat.zip'   # or: git clone <repo URL> /content/spatial-sasp-gat
import shutil, hashlib
if os.path.isdir('/content/spatial-sasp-gat'):
    shutil.rmtree('/content/spatial-sasp-gat')          # never reuse an older unpacked copy: always run the zip that is on Drive
subprocess.run(['unzip', '-q', '-o', REPO_ZIP, '-d', '/content'], check=True)
print('repository zip sha256[:16]:', hashlib.sha256(open(REPO_ZIP, 'rb').read()).hexdigest()[:16], '| size', os.path.getsize(REPO_ZIP))
if 'sasp_gat' in sys.modules: print('WARNING: sasp_gat was already imported in this runtime - reset the runtime so the new code is used')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'anndata'], check=True)   # reads HEST .h5ad; scikit-learn / networkx are preinstalled on Colab (do NOT reinstall: it can swap pandas under a running kernel)
# data_tables exported in ENV-2/ENV-3 (copied only if missing in the repo)
import shutil
for f in ['sasp54_genes_canonical_order.csv', 'ppi_string_v12_edges.csv', 'slide_partition.csv']:
    dst = '/content/spatial-sasp-gat/data_tables/' + f
    if not os.path.exists(dst):
        shutil.copy(BASE + '/code_release_export_2026-10-23/' + f, dst)
sys.path.insert(0, '/content/spatial-sasp-gat/src')
import torch, torch_geometric
print('torch', torch.__version__, '| torch_geometric', torch_geometric.__version__,
      '| cuda', torch.cuda.is_available())

In [ ]:
import hashlib, json, pandas as pd, numpy as np, torch
from pathlib import Path
from sasp_gat.config import *
from sasp_gat.data import load_package, split_samples, sha256_file
from sasp_gat.ppi import ppi_edge_index_from_networkx
from sasp_gat.availability import build_availability, effective_mask, score_mask
from sasp_gat.evaluation import load_model, eval_both
from sasp_gat.training import evaluate_multimask_av

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
REPO = Path('/content/spatial-sasp-gat'); T = REPO / 'data_tables'
PKL = Path(BASE) / 'MODEL_READY_DATASET_v1_2026-09-26.pkl'
pkg, pyg_dataset, gene_order = load_package(PKL)
train_s, val_s, test_s, hd_s = split_samples(pyg_dataset)
AVAIL, LOGGED = build_availability(pyg_dataset.keys(), gene_order, T / 'availability_unmeasured.csv')
ppi_ei = ppi_edge_index_from_networkx(pkg['ppi_graph_networkx'], gene_order, device)
CKDIR = REPO / 'checkpoints' if (REPO / 'checkpoints' / 'full_avail_seed42.pt').exists() else Path(BASE) / 'revision_checkpoints_2026-10-18'
print('checkpoint dir:', CKDIR)
import torch_geometric, scipy; print('versions: torch', torch.__version__, '| pyg', torch_geometric.__version__, '| numpy', np.__version__, '| pandas', pd.__version__, '| scipy', scipy.__version__)

In [ ]:
# 1) checkpoint integrity + stored metadata + validation reproduction
EXPECTED_SHA16 = {42: 'b45e325d3e43f484', 123: 'ce201d4b86d7edd8', 2026: 'eda32cf3ec7265b9'}   # recorded in ENV-1, 2026-10-23
rows = []; MODELS = {}
for seed in SEEDS:
    p = CKDIR / f'full_avail_seed{seed}.pt'
    assert sha256_file(p)[:16] == EXPECTED_SHA16[seed], f'checkpoint hash mismatch: {p.name}'
    m, ck = load_model(p, device, expect_seed=seed, expect_avail=True)
    assert ck['variant_name'] == 'full' and (ck['use_spatial'], ck['use_ppi']) == (True, True)
    assert sum(q.numel() for q in m.parameters()) == N_MODEL_PARAMETERS
    val = evaluate_multimask_av(m, val_s, AVAIL, ppi_ei, device, role='val')
    assert abs(val - float(ck['best_val_loss'])) < 1e-4, (seed, val, ck['best_val_loss'])
    MODELS[seed] = m
    rows.append(dict(seed=seed, val_recomputed=val, val_saved=float(ck['best_val_loss']),
                     best_epoch=ck['best_epoch'], total_epochs=ck['total_epochs_run']))
print(pd.DataFrame(rows).round(6).to_string(index=False))
print('checkpoint hashes, metadata and validation loss reproduced')

In [ ]:
# 2) locked test + external HD, availability-aware score (input_av=True), primary 25% masking
R = []
for seed, m in MODELS.items():
    t = eval_both(m, test_s, AVAIL, ppi_ei, device, 'test', input_av=True)
    h = eval_both(m, hd_s,   AVAIL, ppi_ei, device, 'hd',   input_av=True)
    R.append(dict(seed=seed, test_av=t['mse_av'], hd_av=h['mse_av'], n_test_removed=t['n_old']-t['n_av'],
                  n_test_total=t['n_old'], n_hd_removed=h['n_old']-h['n_av'], n_test_scored=t['n_av'], n_hd_scored=h['n_av']))
R = pd.DataFrame(R); print(R.round(6).to_string(index=False))
assert R.n_test_removed.nunique() == 1 and R.n_hd_removed.max() == 0
ms = lambda s: f'{s.mean():.4f} ± {s.std(ddof=1):.4f}'
print('scored targets per seed: test', R.n_test_scored.tolist(), '| HD', R.n_hd_scored.tolist())
print('TEST  mean ± SD (3 seeds, ddof=1):', ms(R.test_av))
print('HD    mean ± SD (3 seeds, ddof=1):', ms(R.hd_av))
# values recorded in the manuscript (Table 4); a mismatch is reported, not hidden
for name, got, ref in [('test', R.test_av.mean(), 0.1716), ('HD', R.hd_av.mean(), 0.4208)]:
    print(f'{name}: recomputed {got:.4f} vs manuscript {ref:.4f} ->', 'MATCH' if round(got, 4) == ref else 'DIFFERENT')

In [ ]:
# 3) two-hop masking (inference only; locked protocol)
from sasp_gat.twohop import build_graph_cache, masks_2hop, invariants
PROTO = Path(BASE) / 'revision_2hop_2026-10-23' / 'protocol_2hop_v1.json'
txt = PROTO.read_text()
assert hashlib.sha256(txt.encode()).hexdigest()[:16] == TWOHOP_PROTOCOL_SHA256_16, 'protocol file differs from the locked one'
proto = json.loads(txt); assert abs(proto['p_c'] - TWOHOP_P_CENTRE) < 1e-12
GR = build_graph_cache({'test': test_s, 'hd': hd_s}, device)
# graph/availability identifiers recorded in the protocol must match the data used here
def h16(t): return hashlib.sha256(np.ascontiguousarray(t.cpu().numpy()).tobytes()).hexdigest()[:16]
for sid, rec in proto['graph_avail_ids'].items():
    g = GR[sid]
    assert rec['n'] == g['n'] and rec['edge_hash'] == h16(torch.stack([g['src'], g['dst']])) and rec['avail_hash'] == h16(AVAIL[sid])
print('two-hop protocol hash and per-slide graph/availability ids verified')

@torch.no_grad()
def twohop_mse(model, samples, role):
    ss = n = 0
    for sid, d in samples.items():
        x = d.x.to(device); ei = d.edge_index.to(device); av = AVAIL[sid].to(device)
        for i in range(N_EVAL_MASKS):
            hide, score, C, H = masks_2hop(sid, role, i, av, GR[sid], TWOHOP_P_CENTRE, device)
            invariants(sid, hide, score, GR[sid])
            pred = model(x, ei, hide, ppi_ei if model.use_ppi else None).double()
            ss += ((pred - x.double()) ** 2)[score].sum().item(); n += int(score.sum().item())
    return ss / n, n
rows_ = []
for s_, m_ in MODELS.items():
    (tm, tn), (hm, hn) = twohop_mse(m_, test_s, 'test'), twohop_mse(m_, hd_s, 'hd')
    rows_.append(dict(seed=s_, test=tm, hd=hm, n_test_scored=tn, n_hd_scored=hn))
H = pd.DataFrame(rows_)
print(H.round(6).to_string(index=False))
print('TWO-HOP full  test:', ms(H.test), '| HD:', ms(H.hd))
for name, got, ref in [('test', H.test.mean(), 0.2498), ('HD', H.hd.mean(), 0.4964)]:
    print(f'{name}: recomputed {got:.4f} vs manuscript {ref:.4f} ->', 'MATCH' if round(got, 4) == ref else 'DIFFERENT')